In [1]:
import time
import pandas as pd
from pytrends.exceptions import ResponseError

def safe_interest(pytrends, kw_list, timeframe, geo='US', max_retries=3):
    for attempt in range(max_retries):
        try:
            pytrends.build_payload(kw_list, timeframe=timeframe, geo=geo)
            return pytrends.interest_over_time()
        except Exception as e:
            wait = 60 * (attempt + 1)
            print(f"retry {attempt+1} after {wait}s: {e}")
            time.sleep(wait)
    return pd.DataFrame()


# Pytrends: Driver vs. Sponsor Search Interest

Goal: pull Google Trends interest for each driver and their primary sponsor, then check
whether sponsor search interest actually follows driver search interest (lead-lag),
rather than just moving together for unrelated reasons.

`safe_interest` above is the rate-limit-safe wrapper we'll use for every call.

In [2]:
from pytrends.request import TrendReq

pytrends = TrendReq(hl='en-US', tz=360)  # tz=360 = US Central

## Driver → sponsor pairs

`driver_stats_2025-6.csv` has a `Primary_Sponsor` column, but most rows are
`Other/Unknown` (only a handful of drivers have a resolved sponsor). Start from the
rows that *do* have a real sponsor, and add manual overrides for anyone important
that's missing (co-sponsors, sponsors that changed mid-season, etc).

In [3]:
driver_stats = pd.read_csv('../csvs/processed/driver_stats_2025-6.csv')

known = driver_stats[driver_stats['Primary_Sponsor'] != 'Other/Unknown']
driver_sponsor_pairs = dict(zip(known['Driver'], known['Primary_Sponsor']))

# manual overrides / additions for drivers you care about that came back Other/Unknown
driver_sponsor_pairs.update({
    'Denny Hamlin': 'Progressive',
})

driver_sponsor_pairs

{'Austin Hill': "cheddar's Scratch Kitchen",
 'Brad Keselowski': 'Castrol',
 'Denny Hamlin': 'Progressive',
 'Ross Chastain': 'Busch Light',
 'Todd Gilliland': "Love's Travel Stops"}

## Pull interest_over_time for each pair

Driver and sponsor go in the *same* `build_payload` call so their 0-100 scores share a
scale — pulling them separately would make the two series incomparable.

In [4]:
TIMEFRAME = '2025-01-01 2026-06-30'  # match the other 2025-6 processed files

results = {}
for driver, sponsor in driver_sponsor_pairs.items():
    df = safe_interest(pytrends, [driver, sponsor], timeframe=TIMEFRAME)
    if df.empty:
        print(f"no data for {driver} / {sponsor}")
        continue
    results[driver] = df.drop(columns='isPartial', errors='ignore')
    time.sleep(1.5)  # be polite between calls

print(len(results))

retry 1 after 60s: The request failed: Google returned a response with code 429
5


## Does sponsor interest follow driver interest, or the reverse?

For each pair, shift the sponsor series by `k` periods and correlate against the
driver series. If correlation peaks at a *positive* lag (sponsor interest trailing
driver interest by `k` periods), that's evidence driver attention is pulling sponsor
attention along with it, rather than the two moving independently.

Run this on both `results` (daily, `max_lag` in days) and `results_weekly` (weekly,
`max_lag` in weeks) — if both agree on the sign/shape of the lag, that's a much
stronger signal than either alone.

In [6]:
def lead_lag_correlation(df, driver_col, sponsor_col, max_lag=5):
    """Correlate driver[t] with sponsor[t+lag] for lag in -max_lag..max_lag.
    Positive lag = sponsor interest trails driver interest by `lag` periods."""
    lags = range(-max_lag, max_lag + 1)
    corrs = {
        lag: df[driver_col].corr(df[sponsor_col].shift(-lag))
        for lag in lags
    }
    return pd.Series(corrs).sort_index()

def summarize_lead_lag(results_dict, max_lag=5):
    summary = {}
    for driver, df in results_dict.items():
        sponsor_col = [c for c in df.columns if c != driver][0]
        summary[driver] = lead_lag_correlation(df, driver, sponsor_col, max_lag=max_lag)
    out = pd.DataFrame(summary).T
    out['best_lag'] = out.idxmax(axis=1)
    return out

lead_lag_daily = summarize_lead_lag(results, max_lag=5)     # lags in days
#lead_lag_weekly = summarize_lead_lag(results_weekly, max_lag=8)  # lags in weeks

print('daily (short-term):')
display(lead_lag_daily)
#print('weekly (long-term, 5y):')
#display(lead_lag_weekly)

daily (short-term):


,-5,-4,-3,-2,-1,0,1,2,3,4,5,best_lag
Austin Hill,0.018374,0.002880,0.219498,0.257369,0.096496,0.094022,0.014418,0.047014,0.094377,0.098183,0.009126,-2
Brad Keselowski,0.165851,0.104139,0.085099,0.123907,0.137238,0.142098,0.207453,0.256674,0.284278,0.215256,0.185438,3
Denny Hamlin,-0.187125,0.030076,-0.016024,-0.002503,-0.155302,0.034766,0.122994,0.042214,0.044650,0.110313,0.123254,5
Ross Chastain,0.472761,0.493065,0.465530,0.321683,0.309183,0.227381,0.165309,0.116671,0.237590,0.096847,0.090717,-4
Todd Gilliland,-0.129048,-0.101510,-0.065887,0.046106,0.121248,0.050958,-0.028445,-0.198197,-0.161041,-0.167506,-0.161699,-1


In [ ]:
# save raw pulls (both granularities) and the lead-lag summaries alongside your other processed files
for driver, df in results.items():
    safe_name = driver.replace(' ', '_').replace('.', '')
    df.to_csv(f'../csvs/processed/pytrends_daily_{safe_name}_2025-6.csv')

#for driver, df in results_weekly.items():
    #safe_name = driver.replace(' ', '_').replace('.', '')
    #df.to_csv(f'../csvs/processed/pytrends_weekly_{safe_name}_5y.csv')

lead_lag_daily.to_csv('../csvs/processed/pytrends_lead_lag_daily_2025-6.csv')
#lead_lag_weekly.to_csv('../csvs/processed/pytrends_lead_lag_weekly_5y.csv')